# Sports Market MM Scan

Find other Kalshi sports series with the same structural properties as KXNBAPTS:
wide spreads, many fragmented markets, and sufficient volume for two-sided flow.

In [ ]:
import sys, os, time, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
sys.path.insert(0, '.')
from dotenv import load_dotenv; load_dotenv()

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

HOST = "https://api.elections.kalshi.com/trade-api/v2"

def kalshi_get(path, params=None):
    """Simple unauthenticated GET — works for public endpoints."""
    resp = requests.get(f"{HOST}{path}", params=params, timeout=10)
    if resp.status_code == 429:
        time.sleep(2); resp = requests.get(f"{HOST}{path}", params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

print('Ready')

## 1. Target sports prop series

Focus on player prop / stat-based series across all sports — these have the same fragmentation structure as KXNBAPTS (player × game × threshold).

In [ ]:
# Sports player prop and stat series across all leagues
# These share the KXNBAPTS structure: player x game x threshold
SPORTS_SERIES = {
    # NBA (current)
    'KXNBAPTS': 'NBA Player Points',
    'KXNBAREB': 'NBA Player Rebounds',
    'KXNBAAST': 'NBA Player Assists',
    'KXNBA3PT': 'NBA Player Threes',
    'KXNBABLK': 'NBA Player Blocks',
    'KXNBASTL': 'NBA Player Steals',
    'KXNBAGAME': 'NBA Win/Loss',
    'KXNBASPREAD': 'NBA Point Spread',
    'KXNBATOTAL': 'NBA Total Points',
    # MLB
    'KXMLBHRR': 'MLB Hits Runs RBIs',
    'KXMLBTB': 'MLB Total Bases',
    'KXMLBKS': 'MLB Strikeouts',
    'KXMLBTEAMTOTAL': 'MLB Team Total',
    'KXMLBGAME': 'MLB Win/Loss',
    'KXMLBTOTAL': 'MLB Total Runs',
    'KXMLBSPREAD': 'MLB Run Line',
    # NHL
    'KXNHLPTS': 'NHL Player Points',
    'KXNHLGOALS': 'NHL Goals',
    'KXNHLGAME': 'NHL Win/Loss',
    'KXNHLTOTAL': 'NHL Total Goals',
    # NFL
    'KXNFLPTS': 'NFL Player Points',
    'KXNFLPASS': 'NFL Passing Yards',
    'KXNFLRUSH': 'NFL Rushing Yards',
    'KXNFLREC': 'NFL Receiving Yards',
    'KXNFLGAME': 'NFL Win/Loss',
    'KXNFLTOTAL': 'NFL Total Points',
    'KXNFLWINS': 'NFL Season Wins',
    # Soccer
    'KXSOCGOALS': 'Soccer Goals',
    'KXSOCGAME': 'Soccer Match Result',
    # Tennis
    'KXITFMATCH': 'ITF Men Match',
    'KXITFWMATCH': 'ITF Women Match',
    # Golf
    'KXPGATOUR': 'PGA Tour Winner',
    'KXDPWORLDTOUR': 'DP World Tour Winner',
}

# Fetch open market counts for each series
print(f'Scanning {len(SPORTS_SERIES)} sports series...\n')
series_data = []
for ticker, label in SPORTS_SERIES.items():
    try:
        resp = kalshi_get('/markets', params={
            'series_ticker': ticker, 'status': 'open', 'limit': 1000
        })
        markets = resp.get('markets', [])
        n = len(markets)
        if n > 0:
            series_data.append({
                'series': ticker, 'label': label, 'open_markets': n,
                'markets': markets,
            })
            print(f'  {ticker:20s} {label:30s} {n:>4} open markets')
        time.sleep(0.1)
    except Exception as e:
        print(f'  {ticker:20s} ERROR: {e}')

print(f'\n{len(series_data)} series with open markets')

## 2. Sample order books — measure spreads and volume

In [ ]:
deep = []
for sd in series_data:
    series = sd['series']
    markets = sd['markets']
    spreads, volumes = [], []

    for m in markets[:60]:
        ticker = m['ticker']
        vol = m.get('volume', 0) or 0
        volumes.append(vol)
        try:
            ob = kalshi_get(f'/markets/{ticker}/orderbook')['orderbook_fp']
            yes_bids = ob.get('yes_dollars', [])
            no_bids = ob.get('no_dollars', [])
            # yes_dollars: best bid is highest price → first entry after sort
            # no_dollars: best no bid → ask = 1.00 - no_price
            bid = int(round(float(yes_bids[0][0]) * 100)) if yes_bids else 0
            ask = 100 - int(round(float(no_bids[0][0]) * 100)) if no_bids else 100
            if 0 < bid < ask < 100:
                spreads.append(ask - bid)
        except:
            pass
        time.sleep(0.03)

    if not spreads:
        print(f'  {series:20s}  NO BOOKS (0/{len(markets[:60])} had valid bid+ask)')
        continue

    deep.append({
        'series': series,
        'label': sd['label'],
        'open_markets': sd['open_markets'],
        'books_sampled': len(spreads),
        'median_spread': np.median(spreads),
        'mean_spread': np.mean(spreads),
        'p25_spread': np.percentile(spreads, 25),
        'p75_spread': np.percentile(spreads, 75),
        'pct_wide': sum(1 for s in spreads if s >= 4) / len(spreads) * 100,
        'total_volume': sum(volumes),
        'median_volume': np.median(volumes),
        'vol_gt0_pct': sum(1 for v in volumes if v > 0) / len(volumes) * 100,
    })
    d = deep[-1]
    print(f"  {series:20s}  mkts={d['open_markets']:>4}  "
          f"books={d['books_sampled']:>3}  "
          f"spread={d['median_spread']:4.0f}c  "
          f"vol={d['total_volume']:>8}  "
          f"wide={d['pct_wide']:3.0f}%  "
          f"active={d['vol_gt0_pct']:3.0f}%")

ddf = pd.DataFrame(deep)
print(f'\n{len(ddf)} series with order book data')

## 3. Rank and visualize

In [ ]:
if len(ddf) > 0:
    ddf['mm_score'] = (
        ddf['median_spread']
        * np.log1p(ddf['open_markets'])
        * np.log1p(ddf['total_volume'])
        * (ddf['pct_wide'] / 100)
    )
    ddf = ddf.sort_values('mm_score', ascending=False)

    print('All series ranked by MM attractiveness:\n')
    print(ddf[['series', 'label', 'open_markets', 'median_spread', 'total_volume',
               'pct_wide', 'vol_gt0_pct', 'mm_score']].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Bar chart of MM scores
    ax = axes[0]
    colors = ['red' if 'NBA' in r['label'] else 'steelblue' for _, r in ddf.iterrows()]
    ax.barh(range(len(ddf)), ddf['mm_score'], color=colors)
    ax.set_yticks(range(len(ddf)))
    ax.set_yticklabels([f"{r['series']} ({r['label']})" for _, r in ddf.iterrows()], fontsize=7)
    ax.set_xlabel('MM Score')
    ax.set_title('MM Attractiveness (red = NBA)')
    ax.invert_yaxis()

    # Spread vs volume scatter
    ax = axes[1]
    ax.scatter(ddf['total_volume'], ddf['median_spread'],
               s=ddf['open_markets'] * 0.5, alpha=0.6)
    for _, r in ddf.iterrows():
        ax.annotate(r['series'], (r['total_volume'], r['median_spread']),
                    fontsize=6, alpha=0.7)
    ax.set_xlabel('Total volume')
    ax.set_ylabel('Median spread (c)')
    ax.set_title('Spread vs Volume (size = # markets)')
    ax.set_xscale('symlog')

    # % wide spread bars
    ax = axes[2]
    ax.barh(range(len(ddf)), ddf['pct_wide'], color=colors)
    ax.set_yticks(range(len(ddf)))
    ax.set_yticklabels(ddf['series'], fontsize=7)
    ax.set_xlabel('% markets with spread >= 4c')
    ax.set_title('Quotable Market %')
    ax.invert_yaxis()

    plt.suptitle('Sports Series MM Scan', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No data to display')

## 4. Recommendations

In [ ]:
if len(ddf) > 0:
    # Actionable: median_spread >= 3, open_markets >= 20, total_volume >= 100, pct_wide >= 30
    actionable = ddf[
        (ddf['median_spread'] >= 3) &
        (ddf['open_markets'] >= 20) &
        (ddf['total_volume'] >= 100) &
        (ddf['pct_wide'] >= 30)
    ]

    print('=' * 70)
    print('ACTIONABLE CANDIDATES')
    print('(median spread >= 3c, markets >= 20, volume >= 100, wide% >= 30)')
    print('=' * 70)
    if len(actionable) > 0:
        print(actionable[['series', 'label', 'open_markets', 'median_spread',
                          'total_volume', 'pct_wide', 'mm_score']].to_string(index=False))
    else:
        print('  None found meeting all criteria')

    # Also show "close misses" — series that almost qualify
    close = ddf[
        (ddf['median_spread'] >= 2) &
        (ddf['open_markets'] >= 10) &
        (~ddf['series'].isin(actionable['series'] if len(actionable) > 0 else []))
    ]
    if len(close) > 0:
        print(f'\n\nCLOSE MISSES (worth watching):')
        print(close[['series', 'label', 'open_markets', 'median_spread',
                      'total_volume', 'pct_wide', 'mm_score']].to_string(index=False))

    print('\n\nKEY INSIGHT:')
    print('KXNBAPTS works because of player × game × threshold fragmentation.')
    print('Same structure exists in: NBA rebounds/assists/3pt, MLB stats, NFL stats.')
    print('The question is whether those series have enough VOLUME for fills.')
    print('Wide spreads without volume = no fills = no P&L.')
else:
    print('No data')